# Deployment
**Proyek:** Analisis Sentimen Ulasan Shopee — Bummi Tani  
**Tujuan:** Visualisasi hasil analisis sentimen, Word Cloud, komparasi akurasi, dan simulasi prediksi ulasan baru menggunakan model terbaik.

---

## Import Library

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import joblib
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

# NLP (untuk preprocessing ulasan baru)
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Library berhasil diimport.')

## Load Semua Artefak

In [ ]:
PREP_DIR  = '../3-Data-Preparation'
MODEL_DIR = '../4-Modeling'
EVAL_DIR  = '../5-Evaluation'

def load_artifact(filename, base_dir):
    path = os.path.join(base_dir, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f' File tidak ditemukan: {path}')
    obj = joblib.load(path)
    print(f'   Dimuat: {path}')
    return obj

print(' Memuat artefak')

# Data
tfidf_vectorizer = load_artifact('tfidf_vectorizer.pkl', PREP_DIR)

# Model
svm_model = load_artifact('svm_model.pkl', MODEL_DIR)
nb_model  = load_artifact('nb_model.pkl', MODEL_DIR)
knn_model = load_artifact('knn_model.pkl', MODEL_DIR)

# Hasil Evaluasi
eval_csv_path = os.path.join(EVAL_DIR, 'evaluation_results.csv')
eval_results = pd.read_csv(eval_csv_path) if os.path.exists(eval_csv_path) else None

# Dataset bersih
ulasan_bersih_path = os.path.join(PREP_DIR, 'ulasan_bersih.csv')
df_bersih = pd.read_csv(ulasan_bersih_path)
print(f'  Dataset bersih dimuat: {len(df_bersih)} baris')

print(f'\nSemua artefak siap digunakan.')
print(df_bersih.head(3))

## Bar Chart Komparasi Akurasi

In [ ]:
# Data akurasi (dari hasil evaluasi atau manual)
if eval_results is not None:
    model_names = eval_results['Model'].tolist()
    accuracies  = eval_results['Accuracy (%)'].tolist()
else:
    # Fallback: isi manual jika file evaluasi tidak ada
    model_names = ['SVM (Kernel Linear)', 'Multinomial Naïve Bayes', 'KNN']
    accuracies  = [85.0, 80.0, 75.0]  # Ganti dengan nilai aktual

# Sort by accuracy
sorted_pairs = sorted(zip(accuracies, model_names), reverse=True)
accuracies, model_names = zip(*sorted_pairs)

colors_bar = ['#FFD700', '#C0C0C0', '#CD7F32']  # Emas, Perak, Perunggu
if len(model_names) != 3:
    colors_bar = ['#3498DB', '#E67E22', '#27AE60']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, accuracies, color=colors_bar[:len(model_names)],
              edgecolor='white', linewidth=1.2, width=0.5)

# Tambahkan nilai di atas bar
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Styling
ax.set_ylim(0, 110)
ax.set_ylabel('Akurasi (%)', fontsize=12)
ax.set_title('Komparasi Akurasi Algoritma Klasifikasi Sentimen\nSVM vs Naïve Bayes vs KNN',
             fontsize=14, fontweight='bold', pad=15)
ax.axhline(y=80, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='Target 80%')
ax.tick_params(axis='x', labelsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Tambahkan lencana peringkat
rank_labels = [' Terbaik', ' Kedua', ' Ketiga']
for i, (bar, rank) in enumerate(zip(bars, rank_labels[:len(bars)])):
    ax.text(bar.get_x() + bar.get_width()/2., 2,
            rank, ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('komparasi_akurasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: komparasi_akurasi.png')

## Word Cloud — Sentimen Positif & Negatif

In [ ]:
# Pisahkan teks berdasarkan label sentimen
positif_texts = ' '.join(df_bersih[df_bersih['label'] == 1]['ulasan_bersih'].dropna().tolist())
negatif_texts = ' '.join(df_bersih[df_bersih['label'] == 0]['ulasan_bersih'].dropna().tolist())

print(f'Total kata (Positif): {len(positif_texts.split())}')
print(f'Total kata (Negatif): {len(negatif_texts.split())}')

In [ ]:
# Konfigurasi WordCloud
wc_common = dict(
    max_words=100,
    max_font_size=80,
    min_font_size=10,
    background_color='white',
    collocations=False,
    prefer_horizontal=0.8,
    width=800,
    height=400,
)

wc_positif = WordCloud(
    colormap='YlGn',     # Hijau untuk positif
    **wc_common
).generate(positif_texts if positif_texts.strip() else 'tidak ada data positif')

wc_negatif = WordCloud(
    colormap='OrRd',     # Merah-oranye untuk negatif
    **wc_common
).generate(negatif_texts if negatif_texts.strip() else 'tidak ada data negatif')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(wc_positif, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Word Cloud — Ulasan POSITIF',
                   fontsize=14, fontweight='bold', color='#27AE60', pad=12)

axes[1].imshow(wc_negatif, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud — Ulasan NEGATIF',
                   fontsize=14, fontweight='bold', color='#E74C3C', pad=12)

plt.suptitle('Kata-kata Paling Sering Muncul per Kelas Sentimen\n(Ulasan Shopee Bummi Tani)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('wordcloud_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Word Cloud disimpan: wordcloud_sentimen.png')

## Visualisasi Distribusi Sentimen Dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribusi label
dist = df_bersih['sentimen'].value_counts()
colors_pie = ['#27AE60', '#E74C3C']

# Bar Chart
bars = axes[0].bar(dist.index, dist.values, color=colors_pie[:len(dist)],
                    edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribusi Label Sentimen', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Jumlah Ulasan')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

# Pie Chart
axes[1].pie(dist.values, labels=dist.index, autopct='%1.1f%%',
            colors=colors_pie[:len(dist)], startangle=90,
            pctdistance=0.75, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporsi Sentimen', fontsize=12, fontweight='bold')

plt.suptitle('Distribusi Sentimen — Dataset Ulasan Shopee Bummi Tani',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Visualisasi Top 20 Kata per Sentimen

In [ ]:
from collections import Counter

def get_top_words(text, n=20):
    words = text.split()
    counter = Counter(words)
    return pd.DataFrame(counter.most_common(n), columns=['Kata', 'Frekuensi'])

top_pos = get_top_words(positif_texts, 20)
top_neg = get_top_words(negatif_texts, 20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 20 kata Positif
axes[0].barh(top_pos['Kata'][::-1], top_pos['Frekuensi'][::-1],
             color='#27AE60', edgecolor='white', alpha=0.85)
axes[0].set_title('Top 20 Kata — Ulasan Positif', fontsize=12, fontweight='bold', color='#27AE60')
axes[0].set_xlabel('Frekuensi')

# Top 20 kata Negatif
axes[1].barh(top_neg['Kata'][::-1], top_neg['Frekuensi'][::-1],
             color='#E74C3C', edgecolor='white', alpha=0.85)
axes[1].set_title('Top 20 Kata — Ulasan Negatif', fontsize=12, fontweight='bold', color='#E74C3C')
axes[1].set_xlabel('Frekuensi')

plt.suptitle('Kata Paling Sering Muncul per Kelas Sentimen', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik top kata disimpan: top_words.png')

## Setup Fungsi Preprocessing (Ulang untuk Deployment)

In [ ]:
# Inisialisasi tools NLP
stemmer = StemmerFactory().create_stemmer()
stop_words_id = set(stopwords.words('indonesian'))
custom_stopwords = {
    'seller', 'shopee', 'kurir', 'paket', 'pesanan', 'barang',
    'ya', 'yah', 'sih', 'deh', 'dong', 'nih', 'loh', 'lah', 'aja',
    'alhamdulillah', 'semoga', 'aamiin', 'next', 'kali',
}
stop_words_id.update(custom_stopwords)

normalisasi_dict = {
    'baguss': 'bagus', 'bagussss': 'bagus', 'enakk': 'enak', 'murahh': 'murah',
    'cepett': 'cepat', 'gercep': 'cepat', 'oke': 'baik', 'mantap': 'bagus',
    'mantul': 'bagus', 'rekomended': 'direkomendasikan', 'recommended': 'direkomendasikan',
    'packing': 'kemasan', 'seller': 'penjual',
    'tq': 'terima kasih', 'thx': 'terima kasih', 'thanks': 'terima kasih',
    'makasih': 'terima kasih', 'mkasih': 'terima kasih',
    'gak': 'tidak', 'ga': 'tidak', 'ngga': 'tidak', 'nggak': 'tidak',
    'bgt': 'banget', 'udah': 'sudah', 'udh': 'sudah', 'blm': 'belum',
    'krn': 'karena', 'karna': 'karena', 'lg': 'lagi', 'yg': 'yang',
    'dgn': 'dengan', 'jg': 'juga', 'dr': 'dari', 'utk': 'untuk',
    'tp': 'tetapi', 'tpi': 'tetapi', 'kl': 'kalau', 'klo': 'kalau',
    'pake': 'pakai', 'sdh': 'sudah',
}

def preprocessing_pipeline(text):
    """Pipeline preprocessing lengkap untuk deployment."""
    # Case Folding
    text = str(text).lower()
    # Cleaning
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Normalisasi
    words = text.split()
    text = ' '.join([normalisasi_dict.get(w, w) for w in words])
    # Tokenizing
    tokens = word_tokenize(text)
    # Stopword Removal
    tokens = [t for t in tokens if t not in stop_words_id and len(t) > 1]
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

print('Fungsi preprocessing untuk deployment siap.')

## Simulasi Prediksi Ulasan Baru

Masukkan teks ulasan baru pada variabel `ulasan_baru` di bawah ini.

In [ ]:
def prediksi_sentimen(teks_ulasan, model=None, label_model='SVM', verbose=True):
    """
    Fungsi untuk memprediksi sentimen dari teks ulasan baru.

    Parameters:
    -----------
    teks_ulasan : str
        Teks ulasan yang ingin diprediksi sentimennya.
    model : sklearn model
        Model yang sudah dilatih. Default: svm_model.
    label_model : str
        Nama model (untuk tampilan).
    verbose : bool
        Tampilkan detail proses.

    Returns:
    --------
    dict : Berisi teks asli, teks bersih, label, sentimen, dan probabilitas (jika ada).
    """
    if model is None:
        model = svm_model
        label_model = 'SVM (Linear) — Model Terbaik'

    # Step 1: Preprocessing
    teks_bersih = preprocessing_pipeline(teks_ulasan)

    # Step 2: Transform dengan TF-IDF
    X_new = tfidf_vectorizer.transform([teks_bersih])

    # Pastikan non-negatif untuk NB
    X_new_abs = X_new.copy()
    if sp.issparse(X_new_abs):
        X_new_abs.data = np.abs(X_new_abs.data)

    # Step 3: Prediksi
    if 'Bayes' in label_model or 'NB' in label_model:
        label_pred = model.predict(X_new_abs)[0]
    else:
        label_pred = model.predict(X_new)[0]

    sentimen = 'POSITIF' if label_pred == 1 else '❌ NEGATIF'

    if verbose:
        print('─' * 60)
        print(f'  🔍 SIMULASI PREDIKSI SENTIMEN')
        print('─' * 60)
        print(f'  Model         : {label_model}')
        print(f'  Ulasan Asli   : {teks_ulasan[:100]}...' if len(teks_ulasan) > 100 else f'  Ulasan Asli   : {teks_ulasan}')
        print(f'  Teks Bersih   : {teks_bersih[:100]}...' if len(teks_bersih) > 100 else f'  Teks Bersih   : {teks_bersih}')
        print(f'  Jumlah Token  : {len(teks_bersih.split())} kata')
        print(f'  Hasil Prediksi: {sentimen} (label={label_pred})')
        print('─' * 60)

    return {
        'ulasan_asli': teks_ulasan,
        'teks_bersih': teks_bersih,
        'label': label_pred,
        'sentimen': 'Positif' if label_pred == 1 else 'Negatif'
    }

print(' Fungsi prediksi_sentimen() siap.')

In [ ]:
# ============================================================
# 🖊️ MASUKKAN ULASAN BARU DI SINI
# ============================================================
ulasan_baru = "Tepung ketan hitamnya bagus banget! Warnanya hitam pekat, teksturnya lembut. Pengiriman cepat dan packing aman. Pasti order lagi!"

# Prediksi menggunakan SVM (model terbaik)
hasil = prediksi_sentimen(ulasan_baru, model=svm_model, label_model='SVM (Kernel Linear)')

In [ ]:
# Bandingkan prediksi dari ketiga model
print('=' * 60)
print('  PREDIKSI DARI KETIGA MODEL')
print('=' * 60)
print(f'  Ulasan: "{ulasan_baru[:70]}..."')
print()

for model, nama in [(svm_model, 'SVM (Linear)       '),
                     (nb_model,  'Naïve Bayes        '),
                     (knn_model, 'KNN (k=5, cosine)  ')]:
    hasil_temp = prediksi_sentimen(ulasan_baru, model=model, label_model=nama, verbose=False)
    emoji = '✅' if hasil_temp['label'] == 1 else '❌'
    print(f'  {nama}: {emoji} {hasil_temp["sentimen"].upper()}')

print('=' * 60)

In [ ]:
# Uji dengan beberapa contoh ulasan lainnya
test_cases = [
    "Kecewa banget! Tepungnya bocor pas sampai, packing pelit banget. Rugi beli di sini.",
    "Hmm biasa aja sih, belum dicoba tapi pengemasan oke.",
    "Enak banget buat bikin bolu ketan hitam! Wangi dan hasilnya lembut. Recommended!",
    "Lama banget pengirimannya, sudah seminggu baru sampai. Tidak rekomen.",
    "Murah meriah tapi kualitas mantap, hitam pekat legit banget hasilnya.",
]

print('\n' + '='*65)
print('  UJI BATCH PREDIKSI — Model Terbaik (SVM)')
print('='*65)

batch_results = []
for ulasan in test_cases:
    hasil_batch = prediksi_sentimen(ulasan, model=svm_model, label_model='SVM', verbose=False)
    emoji = '✅' if hasil_batch['label'] == 1 else '❌'
    print(f'{emoji} [{hasil_batch["sentimen"]:8s}] "{ulasan[:70]}..."' if len(ulasan) > 70 else f'{emoji} [{hasil_batch["sentimen"]:8s}] "{ulasan}"')
    batch_results.append(hasil_batch)

## Dashboard Ringkasan Visualisasi

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('📊 Dashboard Analisis Sentimen — Ulasan Shopee Bummi Tani',
             fontsize=16, fontweight='bold', y=0.98)

# Layout
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# --- Panel 1: Distribusi Sentimen ---
ax1 = fig.add_subplot(gs[0, 0])
dist = df_bersih['sentimen'].value_counts()
colors_dist = ['#27AE60', '#E74C3C']
bars1 = ax1.bar(dist.index, dist.values, color=colors_dist[:len(dist)], edgecolor='white')
ax1.set_title('Distribusi Sentimen', fontsize=12, fontweight='bold')
ax1.set_ylabel('Jumlah')
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

# --- Panel 2: Akurasi Model ---
ax2 = fig.add_subplot(gs[0, 1])
if eval_results is not None:
    model_names_dash = eval_results['Model'].str.replace('Multinomial ', '').str.replace('(Kernel Linear)', '').str.strip().tolist()
    acc_values = eval_results['Accuracy (%)'].tolist()
else:
    model_names_dash = ['SVM', 'Naïve Bayes', 'KNN']
    acc_values = [85.0, 80.0, 75.0]

bars2 = ax2.bar(model_names_dash, acc_values,
                 color=['#3498DB', '#E67E22', '#27AE60'][:len(model_names_dash)],
                 edgecolor='white')
for bar, val in zip(bars2, acc_values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax2.set_title('Akurasi per Model', fontsize=12, fontweight='bold')
ax2.set_ylabel('Akurasi (%)')
ax2.set_ylim(0, 110)
ax2.tick_params(axis='x', labelsize=9)

# --- Panel 3: Pie Chart Sentimen ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.pie(dist.values, labels=dist.index, autopct='%1.1f%%',
        colors=colors_dist[:len(dist)], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax3.set_title('Proporsi Sentimen', fontsize=12, fontweight='bold')

# --- Panel 4: Top Kata Positif ---
ax4 = fig.add_subplot(gs[1, 0])
top_pos_10 = get_top_words(positif_texts, 10)
ax4.barh(top_pos_10['Kata'][::-1], top_pos_10['Frekuensi'][::-1],
          color='#27AE60', edgecolor='white', alpha=0.85)
ax4.set_title('Top 10 Kata Positif', fontsize=12, fontweight='bold', color='#27AE60')
ax4.set_xlabel('Frekuensi')
ax4.tick_params(axis='y', labelsize=9)

# --- Panel 5: Top Kata Negatif ---
ax5 = fig.add_subplot(gs[1, 1])
top_neg_10 = get_top_words(negatif_texts, 10)
ax5.barh(top_neg_10['Kata'][::-1], top_neg_10['Frekuensi'][::-1],
          color='#E74C3C', edgecolor='white', alpha=0.85)
ax5.set_title('Top 10 Kata Negatif', fontsize=12, fontweight='bold', color='#E74C3C')
ax5.set_xlabel('Frekuensi')
ax5.tick_params(axis='y', labelsize=9)

# --- Panel 6: Info Dataset ---
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
info_text = (
    f'RINGKASAN PROYEK\n'
    f'─────────────────────\n'
    f'Dataset: Shopee Bummi Tani\n'
    f'Total Ulasan: {len(df_bersih)}\n'
    f'Positif: {dist.get("Positif", 0)}\n'
    f'Negatif: {dist.get("Negatif", 0)}\n'
    f'─────────────────────\n'
    f'Model Terbaik: SVM\n'
    f'Metode Labeling: InSet Lexicon\n'
    f'Fitur: TF-IDF (3000 fitur)\n'
    f'Validasi: 10-Fold CV\n'
    f'Imbalance: SMOTE'
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#ECF0F1', alpha=0.8, edgecolor='#BDC3C7'))

plt.savefig('dashboard_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Dashboard disimpan: dashboard_sentimen.png')

## Ringkasan Fase Deployment

| Visualisasi | File |
|-------------|------|
| Bar chart komparasi akurasi | `komparasi_akurasi.png` |
| Word Cloud positif & negatif | `wordcloud_sentimen.png` |
| Top 20 kata per sentimen | `top_words.png` |
| Dashboard lengkap | `dashboard_sentimen.png` |

> **Fungsi Simulasi:** Gunakan `prediksi_sentimen(teks)` untuk memprediksi ulasan baru secara real-time.

```python
# Contoh penggunaan:
prediksi_sentimen("Tepungnya enak banget, pengiriman cepat!")
prediksi_sentimen("Kecewa banget, barang rusak pas sampai.")
```